# 16 — Bidding on Part 2's forecasts (Part 3, Stage 6)

**Does:** firms choose tomorrow's bid on what they know today (Part 2's forecasts); the market then clears on what actually happens. How much of the perfect-information profit do they capture, and do forecast warning signs flag market-power days in advance?

**Input:** Part 2's `part3_forecast_package.csv` (demand median, 10% and 90% per state) and `availability_forecast_test.csv` ("best day-ahead" solar, wind, hydro); Stage 5's equilibrium results. **Runs:** `src/run_part3.py stage6` (~3 min).

**Plan:** `docs/part3_plan.md`, Stage 6. **Logged in:** `docs/model_log.md`, Step 26.

## Settings (Step 26)

| Setting | Value |
|---|---|
| Firms | The two largest, **UP** and **Maharashtra**, each alone (everyone else at cost, as in Stage 4) |
| Strategies compared | **perfect** (best bid for the actual day, upper bound) · **forecast** (best bid on the median forecast) · **robust** (best average profit over the 10%, median and 90% demand forecasts) · **persistence** (best bid for yesterday's actual day) · **cost** (adder 0) |
| Scoring | Each strategy's bid is fixed, then the market clears on the **actual** day: realised profit and consumer cost |
| Warning signs (6.5) | Forecast spare capacity < 15%, forecast near-step flag (Part 2), either; scored against **market-power days** = Stage 5 equilibrium price ≥ 5% above competitive |

**Hypotheses:** forecast bids capture most (> 80%) of the perfect-information gain and beat persistence; robust bids earn a little less on average but lose less on bad days; forecast flags catch market-power days only partly (Stage 3: market power appears on slack days too).

In [ ]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from dispatch import PROCESSED
import run_part3

def load(file, stage):
    """Read a Part 3 result; run its stage first if the file is missing (see src/run_part3.py)."""
    path = PROCESSED / file
    if not path.exists():
        print(f"{file} missing: running {stage} (see src/run_part3.py for run times)")
        run_part3.STAGES[stage]()
    return pd.read_csv(path, parse_dates=["Date"])

structure = pd.read_csv(PROCESSED / "market_structure_daily.csv", parse_dates=["Date"])
national = pd.read_csv(PROCESSED / "market_structure_national_daily.csv", parse_dates=["Date"]).set_index("Date")
crore = 1e7

fc = load("strategic_forecast_daily.csv", "stage6")
eq = load("strategic_eq_national_daily.csv", "stage5")
from forecast import load_forecast_package
pkg = load_forecast_package()
pkg_nat = pkg[pkg["Area"] == "National"].set_index("Date")
print(fc.groupby(["Firm", "Strategy"]).size().unstack())

## 6.2–6.4 Realised profit by strategy

In [ ]:
w = fc.pivot_table(index=["Date", "Firm"], columns="Strategy", values="Profit")
gain = w.sub(w["cost"], axis=0)                     # extra profit over bidding cost
rows = {}
for firm, g in gain.groupby(level="Firm"):
    perfect = g["perfect"].sum()
    rows[firm] = {k: 100 * g[k].sum() / perfect for k in ["forecast", "robust", "persistence"]}
    rows[firm]["Perfect gain (Rs crore/h, avg)"] = g["perfect"].mean() / crore
    rows[firm]["Days forecast bid loses vs cost"] = int((g["forecast"] < -1).sum())
    rows[firm]["Days robust bid loses vs cost"] = int((g["robust"] < -1).sum())
print("Share of the perfect-information gain captured (%):")
display(pd.DataFrame(rows).T.round(1))

c = fc.pivot_table(index=["Date", "Firm"], columns="Strategy", values="Consumer_cost")
print("Consumer cost increase vs everyone at cost (%, average):")
display((100 * c.div(c["cost"], axis=0).sub(1)).groupby(level="Firm").mean().round(2))

worst = gain.groupby(level="Firm").quantile(0.05) / crore
print("Worst 5% of days, gain over cost (Rs crore/h):"); display(worst.round(3))

## 6.5 Do forecast warning signs flag market-power days?

In [ ]:
e = eq.set_index("Date")
power = (e["Price_avg_eq"] / e["Price_avg_cost"] - 1) > 0.05
flags = pd.DataFrame({
    "Spare < 15% (forecast)": pkg_nat["Tightness_forecast"] < 0.15,
    "Near step (forecast)": pkg_nat["Near_step_forecast"].astype(bool),
}).reindex(power.index)
flags["Either"] = flags.any(axis=1)

def pr(flag, truth):
    tp = (flag & truth).sum()
    return {"Flagged days": int(flag.sum()), "Precision %": 100 * tp / max(flag.sum(), 1),
            "Recall %": 100 * tp / max(truth.sum(), 1)}
print(f"Market-power days (equilibrium price >= 5% above competitive): {int(power.sum())} of {len(power)}")
pd.DataFrame({k: pr(v, power) for k, v in flags.items()}).T.round(1)

## Stage 6 conclusions

_(filled in after the run)_